# Interactive Material Coating Inpainting with FLUX LoRA

This notebook provides an interactive interface for running material coating inpainting inference using the trained FLUX LoRA model.

## Setup and Imports

In [ ]:
import os
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
from diffusers.utils import make_image_grid

# Import the inference module and its new API functions
from inference_inpaint_flux_lora import (
    InferenceConfig, 
    setup_inference_pipeline, 
    run_inference_batch,
)

os.chdir("..")
print("All imports loaded successfully!")
print(f"Current working directory: {os.getcwd()}")

## Configuration

In [ ]:
# Create output directory
OUTPUT_DIR = f"inference_inpaint_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Create inference configuration
config = InferenceConfig(
    pretrained_model_name_or_path="black-forest-labs/FLUX.1-Fill-dev",
    lora_weights_path="model-weights/checkpoint-35000",
    resolution=512, #256, #1024
    guidance_scale=30.0,
    num_inference_steps=50,
    seed=43,
    mixed_precision="bf16",
    crop=False,
    output_dir=OUTPUT_DIR
)

print(f"Output directory: {OUTPUT_DIR}")
print("Configuration created!")

## Load Model and Setup Pipeline

In [ ]:
# Setup the complete inference pipeline using the API
print("Setting up inference pipeline...")
pipeline, material_traits_embeddings, device, transforms_dict = setup_inference_pipeline(config)

print(f"Pipeline setup complete!")
print(f"Using device: {device}")
print("Ready for inference!")

## Predefined Image Sets

You can choose from predefined sets of validation images or define your own.

In [ ]:
# Predefined image sets from the original script
ADD_TEXTURE_SOURCE_IMAGES = [
    # "validation_dataset/fire_hydrant/coating_0.png",
    # "validation_dataset/semantic/car/coating_0.jpg",
    # "validation_dataset/semantic/old_furniture/coating_0.png",
    # "validation_dataset/semantic/jeans/coating_0.jpg",
    # "validation_dataset/semantic/interior_office_glass/coating_0.jpg",
    # "validation_dataset/semantic/concrete/coating_0.jpg",
    # "validation_dataset/semantic/old_window/coating_0.png",
    # "validation_dataset/semantic/wooden_bowl/coating_0.jpg",
    # "validation_dataset/semantic/uv_mappings/sphere.jpg",
    # "validation_dataset/semantic/uv_mappings/box.jpg",
    # "validation_dataset/semantic/textured_floor/coating_0.jpg",
    # "validation_dataset/semantic/room_corner/coating_0.jpg",
    # "validation_dataset/semantic/cloth_bag/coating_0.jpg",
    # "validation_dataset/semantic/clay_pot/coating_1_desaturated.jpg",
    # "validation_dataset/semantic/clay_cup/coating_0.jpg",
    # "validation_dataset/semantic/straw_basket/coating_0.png",
    # "validation_dataset/semantic/complex_wooden_pattern/coating_0.jpg",
    # "validation_dataset/semantic/wooden_chess_piece/coating_0.jpg",
    # "validation_dataset/semantic/textured_cloth/coating_0.jpg",
    # "validation_dataset/semantic/textured_cloth/coating_1.jpg",
    # "validation_dataset/semantic/textured_cloth/coating_2.jpg",
    # "validation_dataset/wooden_table/coating_1.png",
    # "validation_dataset/toy/coating_0.jpg",
    # "validation_dataset/wall/coating_0.jpg",
    # "validation_dataset/semantic/bottle/coating_2.jpg",
    # "validation_dataset/semantic/leather_boots/coating_0.jpg",
    # "validation_dataset/semantic/watering_can/coating_0.jpg",
    # "validation_dataset/pawns/coating_0.jpg",
    # "validation_dataset/lion/coating_0.png",
    # "validation_dataset/ceramic/coating_0.jpg",
    # "validation_dataset/pumpkin/coating_0.jpg",
    # "validation_dataset/propane_tank/coating_0.jpg",
    # "validation_dataset/semantic/cutting_board/coating_0.jpg",
    # "validation_dataset/semantic/wooden_hammer/coating_0.jpg",
    # "validation_dataset/semantic/remove_then_add/add_option_2.png",
    "validation_dataset/semantic/remove_then_add/add_option_1.jpg",
    # "validation_dataset/coat_vs_transfer/yoda/coating_0.jpg",
    # "validation_dataset/coat_vs_transfer/marble/marble.jpg",
]

ADD_TEXTURE_COATING_MASKS = [
    # "validation_dataset/fire_hydrant/coating_mask.png",
    # "validation_dataset/semantic/car/mask.png",
    # "validation_dataset/semantic/old_furniture/mask.png",
    # "validation_dataset/semantic/jeans/mask_2.png",
    # "validation_dataset/semantic/interior_office_glass/mask_0.png",
    # "validation_dataset/semantic/concrete/mask.png",
    # "validation_dataset/semantic/old_window/mask_2.png",
    # "validation_dataset/semantic/wooden_bowl/mask.jpg",
    # "validation_dataset/semantic/uv_mappings/sphere_mask.png",
    # "validation_dataset/semantic/uv_mappings/box_mask.png",
    # "validation_dataset/semantic/textured_floor/mask.png",
    # "validation_dataset/semantic/room_corner/mask.png",
    # "validation_dataset/semantic/cloth_bag/mask.jpg",
    # "validation_dataset/semantic/clay_cup/mask_2.jpg",
    # "validation_dataset/semantic/clay_pot/mask_1.png",
    # "validation_dataset/semantic/straw_basket/outside_mask.png",
    # "validation_dataset/semantic/complex_wooden_pattern/mask.png",
    # "validation_dataset/semantic/wooden_chess_piece/coating_0_mask.png",
    # "validation_dataset/semantic/textured_cloth/mask_0.jpg",
    # "validation_dataset/semantic/textured_cloth/mask_1.png",
    # "validation_dataset/semantic/textured_cloth/mask_2.jpg",
    # "validation_dataset/wooden_table/coating_3_mask.png",
    # "validation_dataset/toy/mask.png",
    # "validation_dataset/wall/mask.png",
    # "validation_dataset/semantic/bottle/mask_2.jpg",
    # "validation_dataset/semantic/leather_boots/mask.png",
    # "validation_dataset/semantic/watering_can/mask.png",
    # "validation_dataset/pawns/mask.png",
    # "validation_dataset/lion/coating_0_mask.png",
    # "validation_dataset/ceramic/mask.png",
    # "validation_dataset/pumpkin/coating_0_mask.png",
    # "validation_dataset/propane_tank/mask.png",
    # "validation_dataset/semantic/cutting_board/mask.png",
    # "validation_dataset/semantic/wooden_hammer/mask.png",
    # "validation_dataset/semantic/remove_then_add/add_option_2_mask.png",
    "validation_dataset/semantic/remove_then_add/add_option_1_mask.png",
    # "validation_dataset/coat_vs_transfer/yoda/mask.png",
    # "validation_dataset/coat_vs_transfer/marble/mask.png",
]

ADD_TEXTURE_ALBEDOS = [
    # "validation_dataset/semantic/car/red.png",
    # "validation_dataset/semantic/old_furniture/white.png",
    # "validation_dataset/semantic/jeans/albedo.png",
    # "validation_dataset/semantic/interior_office_glass/albedo_3.png",
    # "validation_dataset/semantic/concrete/albedo_4.jpg",
    # "validation_dataset/semantic/old_window/albedo.jpg",
    # "validation_dataset/semantic/wooden_bowl/albedo.jpg",
    # "validation_dataset/semantic/uv_mappings/albedo_1.jpg",
    # "validation_dataset/semantic/uv_mappings/albedo_2.jpg",
    # "validation_dataset/semantic/textured_floor/white.png",
    # "validation_dataset/semantic/room_corner/albedo_stripes.jpg",
    # "validation_dataset/semantic/cloth_bag/albedo.png",
    # "validation_dataset/semantic/clay_cup/albedo_1.jpg",
    # "validation_dataset/semantic/clay_pot/albedo.png",
    # "validation_dataset/semantic/straw_basket/albedo.png",
    # "validation_dataset/semantic/complex_wooden_pattern/albedo.jpg",
    # "validation_dataset/semantic/wooden_chess_piece/white.png",
    # "validation_dataset/wooden_table/albedo.jpg",
    # "validation_dataset/wall/albedo.jpg",
    # "validation_dataset/semantic/bottle/albedo.jpg",
    # "validation_dataset/semantic/leather_boots/albedo.jpg",
    # "validation_dataset/semantic/watering_can/albedo.jpg",
    # "validation_dataset/pawns/albedo.jpg",
    # "validation_dataset/lion/albedo.jpg",
    # "validation_dataset/ceramic/albedo_2.jpg",
    # "validation_dataset/pumpkin/albedo_uniform.jpg",
    # "validation_dataset/propane_tank/yellow.jpg",
    # "validation_dataset/semantic/cutting_board/white.png",
    # "validation_dataset/semantic/wooden_hammer/albedo.jpg",
    # "validation_dataset/semantic/textured_cloth/albedo_0.jpg",
    # "validation_dataset/semantic/textured_cloth/albedo_1.jpg",
    # "validation_dataset/semantic/textured_cloth/albedo_2.jpg",
    "validation_dataset/semantic/remove_then_add/unwrapped_albedo_512.png",
    # "validation_dataset/coat_vs_transfer/yoda/albedo.jpg",
    # "validation_dataset/coat_vs_transfer/marble/albedo.jpg",
]

# Material fusion examples
MAT_FUSION_SOURCE_IMAGES = [
    "validation_dataset/material_fusion/bunny/coating_0.png",
    "validation_dataset/material_fusion/gray_car/coating_0.png",
    "validation_dataset/material_fusion/pumpkin/coating_0.png",
]

MAT_FUSION_COATING_MASKS = [
    "validation_dataset/material_fusion/bunny/coating_mask.png",
    "validation_dataset/material_fusion/gray_car/coating_mask.png",
    "validation_dataset/material_fusion/pumpkin/coating_mask.png",
]

REMOVE_SOURCE_IMAGES=[
  # "validation_dataset/corroded_pole/coating_0.png",
  # "validation_dataset/painted_floor/coating_0.png",
  # "validation_dataset/painted_wall_1/coating_0.png",
  # "validation_dataset/painted_wall_2/coating_0.png"
  # "validation_dataset/semantic/remove_then_add/remove_option_1.jpg",
  # "validation_dataset/semantic/remove_then_add/remove_option_2.jpg",
    # "validation_dataset/semantic/remove_then_add/remove_option_3.jpg",
  #   "validation_dataset/semantic/remove_then_add/remove_option_4.jpg",
  #   "validation_dataset/semantic/remove_then_add/remove_option_5.jpg",
  # "validation_dataset/semantic/remove_then_add/remove_option_6.jpg",
  "validation_dataset/semantic/remove_then_add/remove_option_7.jpg",
]

REMOVE_COATING_MASKS=[
  # "validation_dataset/corroded_pole/coating_mask.png",
  # "validation_dataset/painted_floor/coating_mask.png",
  # "validation_dataset/painted_wall_1/coating_mask.png",
  # "validation_dataset/painted_wall_2/coating_mask.png"
  # "validation_dataset/semantic/remove_then_add/remove_option_1_mask.png",
  # "validation_dataset/semantic/remove_then_add/remove_option_2_mask.png",
  # "validation_dataset/semantic/remove_then_add/remove_option_3_mask.png",
  # "validation_dataset/semantic/remove_then_add/remove_option_4_mask.jpg",
  # "validation_dataset/semantic/remove_then_add/remove_option_5_mask.png",
  # "validation_dataset/semantic/remove_then_add/remove_option_6.jpg",
  "validation_dataset/semantic/remove_then_add/remove_option_7_mask.jpg",
]

REPLACE_SOURCE_IMAGES=[
  "validation_dataset/corroded_pole/coating_0.png",
  "validation_dataset/painted_floor/coating_0.png",
  "validation_dataset/painted_wall_1/coating_0.png",
]

REPLACE_COATING_MASKS=[
  "validation_dataset/corroded_pole/coating_mask.png",
  "validation_dataset/painted_floor/coating_mask.png",
  "validation_dataset/painted_wall_1/coating_mask.png",
]

print("Predefined image sets loaded!")

In [ ]:
# Select image set - Change this to use different predefined sets
# Options: "ADD_TEXTURE" (with albedos), "MAT_FUSION" (material fusion), "REMOVE", "REPLACE", "UNIFORM"
IMAGE_SET = "ADD_TEXTURE"

if IMAGE_SET == "ADD_TEXTURE":
    source_images = ADD_TEXTURE_SOURCE_IMAGES
    coating_masks = ADD_TEXTURE_COATING_MASKS
    albedos = ADD_TEXTURE_ALBEDOS
elif IMAGE_SET == "UNIFORM":
    source_images = ADD_TEXTURE_SOURCE_IMAGES
    coating_masks = ADD_TEXTURE_COATING_MASKS
    albedos = ["None"] * len(ADD_TEXTURE_SOURCE_IMAGES)
elif IMAGE_SET == "MAT_FUSION":
    source_images = MAT_FUSION_SOURCE_IMAGES
    coating_masks = MAT_FUSION_COATING_MASKS
    albedos = ["None"] * len(MAT_FUSION_SOURCE_IMAGES)  # No albedos for fusion
elif IMAGE_SET == "REMOVE":
    source_images = REMOVE_SOURCE_IMAGES
    coating_masks = REMOVE_COATING_MASKS
    albedos = ["None"] * len(REMOVE_SOURCE_IMAGES)
elif IMAGE_SET == "REPLACE":
    source_images = REMOVE_SOURCE_IMAGES
    coating_masks = REMOVE_COATING_MASKS
    albedos = ["None"] * len(REMOVE_SOURCE_IMAGES)
else:
    raise ValueError(f"Unknown image set: {IMAGE_SET}")

print(f"Using {IMAGE_SET} image set with {len(source_images)} images")
for i, (src, mask, albedo) in enumerate(zip(source_images, coating_masks, albedos)):
    print(f"  {i+1}. {Path(src).name} | {Path(mask).name} | {Path(albedo).name if albedo != 'None' else 'None'}")

    if not os.path.exists(Path(src)) or not os.path.exists(Path(mask)) or (albedo != "None" and not os.path.exists(Path(albedo))):
        raise ValueError("One or more files for this entry do not exist. Please check the paths.")

### Alternative Material Presets

Uncomment one of the sections below to use different material presets:

In [ ]:
# # Thickness variations
# base_colors = ["0.9,0.8,0.1", "0.9,0.8,0.1", "0.9,0.8,0.1", "0.9,0.8,0.1"]
# thicknesses = [0.0, 0.01, 0.5, 1.0]
# metallics = [1.0, 1.0, 1.0, 1.0]
# roughnesses = [0.0, 0.0, 0.0, 0.0]
# transmission_weights = [0.0, 0.0, 0.0, 0.0]
# uv_options = ["cubic", "cubic", "cubic", "cubic"] # cubic | spherical | original | planar
# task_options = ["AT", "AT", "AT", "AT"]

# # Roughness variations
# base_colors = ["0.9,0.8,0.1", "0.9,0.8,0.1", "0.9,0.8,0.1", "0.9,0.8,0.1"]
# thicknesses = [1.0, 1.0, 1.0, 1.0]
# metallics = [1.0, 1.0, 1.0, 1.0]
# roughnesses = [0.0, 0.01, 0.5, 1.0]
# transmission_weights = [0.0, 0.0, 0.0, 0.0]
# uv_options = ["cubic", "cubic", "cubic", "cubic"] # cubic | spherical | original | planar
# task_options = ["AT", "AT", "AT", "AT"]

# # Color variations
# base_colors = ["0.9,0.8,0.1", "0.2,0.8,0.3", "0.3,0.4,1.0"]
# thicknesses = [0.5, 0.5, 0.5]
# metallics = [0.0, 0.0, 0.0]
# roughnesses = [1.0, 1.0, 1.0]
# transmission_weights = [0.0, 0.0, 0.0]
# uv_options = ["cubic", "cubic", "cubic"] # cubic | spherical | original | planar
# task_options = ["AU", "AU", "AU"]

# # Remove coating examples
# base_colors = ["0.0,0.0,0.0"]
# thicknesses = [0.0]
# metallics = [0.0]
# roughnesses = [0.0]
# transmission_weights = [0.0]
# uv_options = ["spherical"] # cubic | spherical | original | planar
# task_options = ["RM"]

# Replace task
# base_colors=["0.9,0.8,0.1", "0.9,0.8,0.1", "0.7,0.8,1.0"]
# thicknesses=(1.0, 0.0, 0.0)
# metallics=(1.0, 1.0, 0.0)
# roughnesses=(0.0, 0.8, 0.0)
# transmission_weights=(0.0, 0.0, 1.0)
# uv_options = ["cubic", "cubic", "cubic"] # cubic | spherical | original | planar
# task_options=("RL", "RL", "RL")

# Test single material
# base_colors = ["0.9,0.8,0.1"]
# thicknesses = [0.0]
# metallics = [0.0]
# roughnesses = [0.0]
# transmission_weights = [0.0]
# uv_options = ["spherical"] # cubic | spherical | original | planar
# task_options = ["AU"]

# Compare metallic / dielectric
# base_colors = ["0.9,0.8,0.1"]
# thicknesses = [0.0, 0.0]
# metallics = [1.0, 0.0]
# roughnesses = [0.0, 0.0]
# transmission_weights = [0.0, 0.0]
# uv_options = ["cubic", "cubic"] # cubic | spherical | original | planar
# task_options = ["AT", "AT"]

# base_colors= ["0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.9,0.8,0.1", "0.3,0.4,1.0"]
# thicknesses=         [0.0, 0.5, 0.0, 0.5, 0.0, 0.0]
# metallics=           [1.0, 1.0, 0.0, 0.0, 0.0, 0.0]
# roughnesses=         [0.0, 0.3, 1.0, 0.0, 0.0, 1.0]
# transmission_weights=[0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
# task_options=        ("AT", "AT", "AT", "AT", "AU", "AU")
# uv_options = ["cubic", "cubic", "cubic", "cubic", "cubic", "cubic"] # cubic | spherical | original | planar

# Semantics: [ car | old_furniture | jeans | interior_office_glass | concrete | textured_floor | room_corner | cloth_bag | clay_pot | straw_basket ]

# base_colors= ["0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0", "0.0,0.0,0.0" ]
# thicknesses=         [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0]
# metallics=           [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
# roughnesses=         [0.0, 0.0, 1.0, 0.3, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
# transmission_weights=[0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
# task_options=        ("AT", "AT", "AT", "AT", "AT", "AT", "AT", "AT", "AT", "AT")
# uv_options = ["cubic", "cubic", "cubic", "cubic", "cubic", "cubic", "cubic", "cubic", "cubic", "cubic"] # cubic | spherical | original | planar


# base_colors = ["0.9,0.2,1.0", "0.9,0.2,1.0", "0.9,0.2,1.0"]
# thicknesses = [1.0, 1.0, 1.0]
# metallics = [0.0, 0.0, 0.0]
# roughnesses = [0.0, 0.5, 1.0]
# transmission_weights = [0.0, 0.0, 0.0]
# uv_options = ["cubic", "cubic", "cubic"] # cubic | spherical | original | planar
# task_options = ["AT", "AT", "AT"]

# base_colors = ["0.0,1.0,0.0", "0.0,1.0,0.0"]
# thicknesses = [0.0, 1.0]
# metallics = [0.0, 0.0]
# roughnesses = [1.0, 1.0]
# transmission_weights = [0.0, 0.0]
# uv_options = ["spherical", "spherical"] # cubic | spherical | original | planar
# task_options = ["AT", "AT"]

# Test single material from the list
base_colors = ["0.7,0.6,0.1"]
thicknesses = [0.0]
metallics = [0.0]
roughnesses = [1.0]
transmission_weights = [0.0]
uv_options = ["cubic"] # cubic | spherical | original | planar
task_options = ["AT"]

print("Material preset options available (uncomment to use)")

## Run Inference

In [ ]:
print(f"Running inference with {len(source_images)} images and {len(base_colors)} material variations...")

for i in range(50):
    config.seed = i
    all_results = run_inference_batch(
        pipeline=pipeline,
        material_traits_embeddings=material_traits_embeddings,
        device=device,
        transforms_dict=transforms_dict,
        config=config,
        source_images=source_images,
        coating_masks=coating_masks,
        albedos=albedos,
        base_colors=base_colors,
        uv_options=uv_options,
        thicknesses=thicknesses,
        metallics=metallics,
        roughnesses=roughnesses,
        transmission_weights=transmission_weights,
        task_options=task_options
    )

print(f"\nGeneration complete! Generated {len(all_results)} images successfully.")

## Visualize Results

In [ ]:
# Create and save complete results grid
# if all_results:
#     print("Creating complete results grid...")

#     # Create grid with 4 columns: source, mask, projected albedo, generated
#     grid_images = []
#     for result in all_results:
#         row_images = [
#             result['source_image'].resize((config.resolution // 2, config.resolution // 2)),
#             result['mask_image'].resize((config.resolution // 2, config.resolution // 2)),
#             result['projected_albedo'].resize((config.resolution // 2, config.resolution // 2)),
#             result['generated_image'].resize((config.resolution // 2, config.resolution // 2))
#         ]
#         grid_images.extend(row_images)

#     # Create final grid: 4 columns, with rows = total_combinations
#     if grid_images:
#         final_grid = make_image_grid(grid_images, rows=len(all_results), cols=4)

#         grid_path = os.path.join(OUTPUT_DIR, "results_grid.png")
#         final_grid.save(grid_path)
#         print(f"Results grid saved to: {grid_path}")

#         # Display the grid
#         plt.figure(figsize=(16, len(all_results) * 3))
#         plt.imshow(final_grid)
#         plt.axis('off')
#         plt.title('Complete Results Grid\n(Source | Mask | Projected Albedo | Generated)')
#         plt.show()
#     else:
#         print("No images to create grid")
# else:
#     print("No results to create grid")

## Summary

In [ ]:
# print("=== INFERENCE SUMMARY ===")
# if all_results:
#     print(f"Total images generated: {len(all_results)}")
#     print(f"Source images used: {len(source_images)}")
#     print(f"Material variations: {len(base_colors)}")
#     print(f"Output directory: {OUTPUT_DIR}")
#     if os.path.exists(os.path.join(OUTPUT_DIR, "results_grid.png")):
#         print(f"Results grid: {os.path.join(OUTPUT_DIR, 'results_grid.png')}")
#     print("\\nGenerated files:")
#     for result in all_results:
#         if result['output_path']:
#             print(f"  - {Path(result['output_path']).name}")
        
#     print("\\nMaterial descriptions:")
#     for i, result in enumerate(all_results):
#         print(f"  {i+1}. {Path(result['source_path']).stem}: {result['material_description']}")
# else:
#     print("No results generated")
    
# print("\n=== NOTEBOOK COMPLETED ===")